# 10 — Sister Cities demo → graph GeoAI

**[🚀 Launch this notebook live](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=10_sister_cities_graph_geoai.ipynb)**

This lab reuses the actual features embedded in GeoLibre’s [Sister Cities of Major World Cities](https://share.geolibre.app/giswqs/sister-cities-of-major-world-cities) demo. Its 1,108 international relationships become a graph: cities are nodes, partnerships are edges, and edge length is the great-circle distance supplied by the demo.

We derive graph features, cluster cities by network role, flag unusual connectivity profiles, and send the results back to GeoLibre. This is exploratory graph GeoAI—not a ranking of cities or a claim about diplomatic importance.

In [ ]:
import sys
if sys.platform == 'emscripten':
    import micropip
    await micropip.install(['geolibre==3.0.0', 'pyodide-http', 'networkx'])
    import pyodide_http
    pyodide_http.patch_all()

import requests
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from IPython.display import IFrame, display
from geolibre_lite import LiteMap as Map

assert Map.__module__ == 'geolibre_lite'

## Compare with the authored demo

The live project uses great-circle arcs colored by distance and points sized by the number of twin-city links. Pan, inspect popups, and then continue below to compute new analytical attributes from the same project file.

In [ ]:
DEMO = 'https://share.geolibre.app/giswqs/sister-cities-of-major-world-cities'
PROJECT_JSON = DEMO + '.geolibre.json'
display(IFrame(DEMO + '?embed=1', width='100%', height=620))

In [ ]:
response = requests.get(PROJECT_JSON, timeout=90)
response.raise_for_status()
project = response.json()

links_fc = next(layer['geojson'] for layer in project['layers'] if 'links' in layer['name'].lower())
cities_fc = next(layer['geojson'] for layer in project['layers'] if 'cities' in layer['name'].lower())
print(project['name'])
print(f"{len(cities_fc['features']):,} mapped cities; {len(links_fc['features']):,} partnerships")
print(project['metadata']['description'])

## Build the spatial network

The city-point layer supplies coordinates and a reported link count. The relationship layer supplies `from`, `to`, and distance in kilometres. Approximate betweenness uses a fixed sample and seed so the browser calculation is fast and reproducible.

In [ ]:
locations = {}
for feature in cities_fc['features']:
    p = feature['properties']
    key = f"{p['city']}, {p['country']}"
    lon, lat = feature['geometry']['coordinates'][:2]
    locations[key] = {'lon': lon, 'lat': lat, 'reported_twins': p['twins']}

G = nx.Graph()
for feature in links_fc['features']:
    p = feature['properties']
    G.add_edge(p['from'], p['to'], km=float(p['km']))

betweenness = nx.betweenness_centrality(G, k=min(100, len(G)), seed=42, weight='km')
rows = []
for city in G.nodes:
    edge_distances = [d['km'] for _, _, d in G.edges(city, data=True)]
    loc = locations.get(city, {})
    rows.append({
        'city': city,
        'degree': G.degree(city),
        'betweenness': betweenness[city],
        'mean_link_km': np.mean(edge_distances),
        'max_link_km': np.max(edge_distances),
        'lon': loc.get('lon'),
        'lat': loc.get('lat'),
    })
city_features = pd.DataFrame(rows)
city_features.sort_values('degree', ascending=False).head(10)

## Cluster network roles and detect unusual profiles

The model uses degree, approximate betweenness, mean link distance, and maximum link distance. Log transforms reduce the leverage of a few very large values. Isolation Forest then highlights combinations that are uncommon in this particular dataset.

In [ ]:
feature_columns = ['degree', 'betweenness', 'mean_link_km', 'max_link_km']
X = np.log1p(city_features[feature_columns].to_numpy())
Xs = StandardScaler().fit_transform(X)

city_features['network_role'] = KMeans(n_clusters=4, n_init=20, random_state=42).fit_predict(Xs)
detector = IsolationForest(n_estimators=250, contamination=0.06, random_state=42)
city_features['unusual_profile'] = detector.fit_predict(Xs) == -1
city_features['anomaly_score'] = -detector.score_samples(Xs)

role_summary = city_features.groupby('network_role')[feature_columns].median().round(2)
role_summary['cities'] = city_features.groupby('network_role').size()
display(role_summary)
display(city_features.nlargest(12, 'anomaly_score')[['city'] + feature_columns + ['network_role', 'anomaly_score']])

## Return the model output to GeoLibre

Each cluster is a separate styled point layer. The final layer enlarges the unusual profiles, so the statistical result can be inspected against geography. Cities without a matching point in the authored demo remain in the table but are omitted from the map.

In [ ]:
mapped = city_features.dropna(subset=['lon', 'lat']).copy()
palette = ['#3b82f6', '#10b981', '#f59e0b', '#8b5cf6']
m = Map(center=(10, 25), zoom=1.3, height='700px')

def point_collection(frame):
    features = []
    for row in frame.itertuples(index=False):
        props = {
            'city': row.city, 'degree': int(row.degree),
            'betweenness': round(float(row.betweenness), 5),
            'mean_link_km': round(float(row.mean_link_km)),
            'network_role': int(row.network_role),
            'anomaly_score': round(float(row.anomaly_score), 4),
        }
        features.append({'type': 'Feature', 'properties': props, 'geometry': {'type': 'Point', 'coordinates': [row.lon, row.lat]}})
    return {'type': 'FeatureCollection', 'features': features}

for role, color in enumerate(palette):
    subset = mapped[mapped['network_role'] == role]
    m.add_geojson(point_collection(subset), name=f'Network role {role}', pointColor=color, pointRadius=5, pointOpacity=0.8)

outliers = mapped[mapped['unusual_profile']]
m.add_geojson(point_collection(outliers), name='Unusual network profiles', pointColor='#ef4444', pointRadius=10, strokeColor='#ffffff', strokeWidth=2)
m

## Interpretation and responsible use

- A high degree or betweenness value describes this extracted graph, not the overall international importance of a city.
- Missing or uneven Wikidata coverage can produce apparent patterns.
- Great-circle distance is not travel time, cultural proximity, or relationship strength.
- Isolation Forest marks rare feature combinations; it does not identify errors or make causal claims.

## Data & software citations

- [GeoLibre Sister Cities demo](https://share.geolibre.app/giswqs/sister-cities-of-major-world-cities) and its [portable project JSON](https://share.geolibre.app/giswqs/sister-cities-of-major-world-cities.geolibre.json).
- Structured relationship data: [Wikidata](https://www.wikidata.org/wiki/Wikidata:Data_access), released under CC0.
- GeoLibre: https://geolibre.app/ — MIT-licensed software.
- NetworkX and scikit-learn provide the graph metrics and exploratory models.